# E4 — Multi-seed: mean ± std cho YOLO26s (chuẩn thống kê Q1)
Train lại YOLO26s cấu hình thắng (lr0=0.01, optimizer auto, 30 epochs, RFS) với **3 seed mới (0, 1, 2)**;
gộp với seed 42 đã có (mAP50_test 0.789) → mean ± std trên 4 seeds.

**Add Input:** dataset `ppe-unified-v1` (KHÔNG cần notebook nào). GPU T4 x2 · Internet ON.
Ước tính ~8.5-9h (3 × ~2.8h). Có checkpoint-resume: nếu bị cắt 12h, Add Input output phiên cũ rồi Run All.

In [ ]:
# ==================== CONFIG (E4) ====================
QUICK_TEST = False
SEED = 42                      # seed cho copy dữ liệu (cố định); seed train thay đổi theo thí nghiệm
RFS_T, REPEAT_CAP = 0.15, 4

import os, random, shutil, glob, json, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
# RESUME: tự nạp checkpoint từ output phiên trước của CHÍNH notebook này (nếu có Add Input)
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith("e4_") for k in _s):
        shutil.copy(_st, STATE_PATH)
        _src_runs = os.path.join(os.path.dirname(_st), "runs")
        if os.path.isdir(_src_runs): shutil.copytree(_src_runs, f"{WORK}/runs", dirs_exist_ok=True)
        print("ĐÃ NẠP CHECKPOINT từ:", _st); break
STATE = json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), indent=1, default=str)
def done(k): return k in STATE
def mark(k, v): STATE[k] = v; save_state()
def free():
    import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("[E4] stage đã xong:", list(STATE) or "(chưa có)")

In [ ]:
%pip install -q ultralytics
import torch; assert torch.cuda.is_available(), "Bật GPU T4 x2!"
cap = torch.cuda.get_device_capability(0); assert cap[0] >= 7, "GPU quá cũ - chọn T4 x2!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")

In [ ]:
from pathlib import Path
from collections import Counter
from math import sqrt
import yaml

def prepare_dataset(dst_name, use_rfs=True):
    """Copy PPE_Unified_v1 từ /kaggle/input về working; use_rfs=True thì oversample lớp hiếm (train)."""
    DS_ROOT = None
    for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
        root = os.path.dirname(c)
        if os.path.isdir(f"{root}/train/images") and os.path.isdir(f"{root}/val/images") and os.path.isdir(f"{root}/test/images"):
            DS_ROOT = root; break
    assert DS_ROOT, "Chưa Add Input dataset ppe-unified-v1!"
    raw = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))["names"]
    if isinstance(raw, dict): raw = [raw[k] for k in sorted(raw)]
    CLASSES = list(raw); NUM_CLS = len(CLASSES)

    DATA = f"{WORK}/{dst_name}"
    splits = {}
    for sp in ["train", "val", "test"]:
        os.makedirs(f"{DATA}/{sp}/images", exist_ok=True); os.makedirs(f"{DATA}/{sp}/labels", exist_ok=True)
        files = sorted(os.listdir(f"{DS_ROOT}/{sp}/images"))
        if len(os.listdir(f"{DATA}/{sp}/images")) < len(files):
            for fn in files:
                shutil.copy(f"{DS_ROOT}/{sp}/images/{fn}", f"{DATA}/{sp}/images/{fn}")
                lp = f"{DS_ROOT}/{sp}/labels/{Path(fn).stem}.txt"
                if os.path.exists(lp): shutil.copy(lp, f"{DATA}/{sp}/labels/{Path(fn).stem}.txt")
        splits[sp] = files

    if use_rfs:
        img_classes = {}
        for lf in os.listdir(f"{DATA}/train/labels"):
            if "_rfs" in lf: continue
            present = Counter()
            for line in open(f"{DATA}/train/labels/{lf}"):
                if line.strip(): present[int(line.split()[0])] += 1
            img_classes[lf] = present
        n_train = len(splits["train"])
        f_c = {c: sum(1 for s in img_classes.values() if c in s)/max(1, n_train) for c in range(NUM_CLS)}
        r_c = {c: max(1.0, sqrt(RFS_T/max(f_c[c], 1e-6))) for c in range(NUM_CLS)}
        added = 0
        for lf, present in img_classes.items():
            if not present: continue
            r = min(REPEAT_CAP, max(r_c[c] for c in present))
            if r < 1.5: continue
            stem = Path(lf).stem
            imgs = glob.glob(f"{DATA}/train/images/{stem}.*")
            if not imgs: continue
            ext = Path(imgs[0]).suffix
            for k in range(int(round(r)) - 1):
                dst = f"{DATA}/train/images/{stem}_rfs{k}{ext}"
                if not os.path.exists(dst):
                    shutil.copy(imgs[0], dst)
                    shutil.copy(f"{DATA}/train/labels/{lf}", f"{DATA}/train/labels/{stem}_rfs{k}.txt")
                    added += 1
        print(f"  RFS: +{added} bản sao lớp hiếm")

    names_yaml = "\n".join(f"  {i}: {nm}" for i, nm in enumerate(CLASSES))
    with open(f"{DATA}/data.yaml", "w") as f:
        f.write(f"path: {DATA}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n{names_yaml}\n")
    print(f"{dst_name}: train={len(glob.glob(f'{DATA}/train/images/*'))} (use_rfs={use_rfs})")
    return f"{DATA}/data.yaml", CLASSES

In [ ]:
DYAML, CLASSES = prepare_dataset("dataset", use_rfs=True)

## Train 3 seeds

In [ ]:
from ultralytics import YOLO
import pandas as pd
SEEDS = [0, 1, 2]
rows = [{"seed": 42, "mAP50_test": 0.789, "mAP50_95_test": 0.4851, "note": "từ notebook 01 (tham chiếu)"}]
for sd in SEEDS:
    key = f"e4_seed{sd}"
    if done(key): rows.append(STATE[key]); print("skip:", STATE[key]); continue
    t0 = time.time()
    m = YOLO("yolo26s.pt")
    last = f"{WORK}/runs/e4_seed{sd}/weights/last.pt"
    if os.path.exists(last):
        print("RESUME train từ", last); m = YOLO(last); m.train(resume=True)
    else:
        m.train(data=DYAML, epochs=30, imgsz=640, batch=16, seed=sd, lr0=0.01,
                project=f"{WORK}/runs", name=f"e4_seed{sd}", exist_ok=True, plots=False, verbose=False)
    t = m.val(data=DYAML, split="test", verbose=False)
    row = {"seed": sd, "mAP50_test": round(t.box.map50, 4), "mAP50_95_test": round(t.box.map, 4),
           "min": round((time.time() - t0) / 60, 1)}
    rows.append(row); mark(key, row); print(row); del m; free()

df = pd.DataFrame(rows); display(df)
import numpy as np
stat = {"n_seeds": len(df),
        "mAP50_mean": round(float(df.mAP50_test.mean()), 4), "mAP50_std": round(float(df.mAP50_test.std(ddof=1)), 4),
        "mAP50_95_mean": round(float(df.mAP50_95_test.mean()), 4), "mAP50_95_std": round(float(df.mAP50_95_test.std(ddof=1)), 4)}
print("KẾT QUẢ E4:", stat)
df.to_csv(f"{WORK}/results/e4_seeds.csv", index=False)
json.dump(stat, open(f"{WORK}/results/e4_stats.json", "w"), indent=1)

In [ ]:
# ==================== DỌN OUTPUT: KHÔNG kèm dataset ====================
# Giữ lại: results/, state.json, runs/ (cần cho resume các seed). Xóa mọi thứ khác
# (dataset đã copy, yolo26s.pt pretrained tải về, cache...).
KEEP = {"results", "state.json", "runs"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)

def du(path):
    t = 0
    for r, _, fs in os.walk(path):
        t += sum(os.path.getsize(os.path.join(r, f)) for f in fs)
    return t
print("Output cuối:", sorted(os.listdir(WORK)), f"| tổng {du(WORK)/1e6:.1f} MB (không có dataset)")